# Cross-Dataset Generalization Matrix
Evaluates every trained model against every test set:

| Train model | Run1 test | Run2 test | DeepPCB test |
|---|---|---|---|
| **Run1** (HRIPCB-local) | in-domain ✓ | **new** | **new** |
| **Run2** (Kaggle-aug) | **new** | in-domain ✓ | 0.000 ✓ |

**Key detail:** Run1 and Run2 were trained with *different class-index orders*.  
Before each cross-evaluation the GT labels are remapped so indices match the  
evaluating model's training space. Mixing up that remap silently zeroes the metrics.

## 1. Environment

In [ ]:
import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

In [ ]:
import subprocess, sys
try:
    import ultralytics, kagglehub, yaml
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","ultralytics","kagglehub","pyyaml","-q"], check=True)
    import ultralytics, kagglehub, yaml
print("ultralytics:", ultralytics.__version__)

## 2. Configuration

All paths are resolved relative to the project root at runtime.

In [ ]:
from pathlib import Path
import os

_cwd = Path(os.getcwd()).resolve()
ROOT_ENV = os.environ.get("PCB_PROJECT_ROOT")
if ROOT_ENV:
    PROJECT_ROOT = Path(ROOT_ENV).resolve()
elif _cwd.name == "experiments":
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "experiments").exists():
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = _cwd

# ── model weights (git-ignored — must be your local trained files) ──
RUN1_WEIGHTS = PROJECT_ROOT / "results" / "exp_001_yolov11n_baseline_200ep"       / "weights" / "best.pt"
RUN2_WEIGHTS = PROJECT_ROOT / "results" / "exp_002_yolov11n_kaggle_pcb_200ep"     / "weights" / "best.pt"

SEED, IMG_SIZE, BATCH, DEVICE = 42, 640, 16, 0   # DEVICE='cpu' if no GPU

# ── class-name order each model was trained with ──
RUN1_NAMES = ['missing_hole','mouse_bite','open_circuit','short','spur','spurious_copper']
RUN2_NAMES = ['mouse_bite','spur','missing_hole','short','open_circuit','spurious_copper']

# index remaps between the two spaces (computed from the name lists)
RUN2_TO_RUN1 = {i: RUN1_NAMES.index(RUN2_NAMES[i]) for i in range(6)}
RUN1_TO_RUN2 = {i: RUN2_NAMES.index(RUN1_NAMES[i]) for i in range(6)}

RESULTS_DIR  = PROJECT_ROOT / "results";    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CROSS_TMP    = PROJECT_ROOT / "cross_tmp";  CROSS_TMP.mkdir(parents=True, exist_ok=True)
DEEPPCB_YOLO = PROJECT_ROOT / "deeppcb_yolo"

print("Project root:", PROJECT_ROOT)
print("Run1 weights :", RUN1_WEIGHTS.exists(), "->", RUN1_WEIGHTS)
print("Run2 weights :", RUN2_WEIGHTS.exists(), "->", RUN2_WEIGHTS)
print("RUN2_TO_RUN1 :", RUN2_TO_RUN1)
print("RUN1_TO_RUN2 :", RUN1_TO_RUN2)

## 3. Load models and verify class-index order

If either check says MISMATCH, stop and fix `RUN1_NAMES` / `RUN2_NAMES` before proceeding.

In [ ]:
from ultralytics import YOLO

def load_and_verify(weights, expected_names, label):
    assert weights.exists(), f"{label} weights not found at {weights}. These are git-ignored — use your local best.pt."
    m = YOLO(str(weights))
    actual = [m.names[i] for i in range(len(m.names))]
    ok = (actual == expected_names)
    status = "verified ✔" if ok else f"MISMATCH — fix {label}_NAMES before evaluating!"
    print(f"{label} ({status})\n  expected : {expected_names}\n  actual   : {actual}\n")
    return m

model_run1 = load_and_verify(RUN1_WEIGHTS, RUN1_NAMES, "Run1")
model_run2 = load_and_verify(RUN2_WEIGHTS, RUN2_NAMES, "Run2")

## 4. Locate Run 2 test data (Kaggle dataset)

Uses `kagglehub` — returns the cached path instantly if already downloaded.

In [ ]:
import yaml as _yaml

print("Fetching Kaggle dataset path ...")
kaggle_root = kagglehub.dataset_download("norbertelter/pcb-defect-dataset")
KAG_DIR       = Path(kaggle_root) / "pcb-defect-dataset"
KAG_TEST_IMGS = KAG_DIR / "test" / "images"
KAG_TEST_LBLS = KAG_DIR / "test" / "labels"

assert KAG_TEST_IMGS.exists(), f"Kaggle test images not found at {KAG_TEST_IMGS}"

# Confirm the Kaggle yaml class order matches what we expect for Run2
kag_yaml = _yaml.safe_load((KAG_DIR / "data.yaml").read_text())
kag_names_raw = kag_yaml["names"]
kag_names = [kag_names_raw[i] for i in range(6)] if isinstance(kag_names_raw,dict) else list(kag_names_raw)
if kag_names != RUN2_NAMES:
    print("WARNING: Kaggle yaml class order differs from RUN2_NAMES!", kag_names)
else:
    print("Kaggle yaml class order matches RUN2_NAMES ✔")

print(f"Run2 test: {len(list(KAG_TEST_IMGS.glob('*')))} images | "
      f"{len(list(KAG_TEST_LBLS.glob('*.txt')))} label files")

## 5. Locate Run 1 test data

Built by `run1.ipynb` into `pcb_yolo_dataset/`. Run that notebook first if the directory is missing.

In [ ]:
RUN1_TEST_IMGS = PROJECT_ROOT / "pcb_yolo_dataset" / "test" / "images"
RUN1_TEST_LBLS = PROJECT_ROOT / "pcb_yolo_dataset" / "test" / "labels"
assert RUN1_TEST_IMGS.exists(), (
    f"Run1 test images not found at {RUN1_TEST_IMGS}. "
    "Run run1.ipynb first to build pcb_yolo_dataset/."
)
print(f"Run1 test: {len(list(RUN1_TEST_IMGS.glob('*')))} images | "
      f"{len(list(RUN1_TEST_LBLS.glob('*.txt')))} label files")

deeppcb_ok = (DEEPPCB_YOLO / "test" / "images").exists()
print(f"DeepPCB data found: {deeppcb_ok} -> {DEEPPCB_YOLO}")
if not deeppcb_ok:
    print("  (run3_crossdataset_benchmark_deeppcb.ipynb first to enable Run1->DeepPCB row)")

## 6. Helper functions

`remap_labels` rewrites class indices in every GT label file.  
`link_images` hardlinks (or copies) images without duplicating bytes.  
`write_yolo_yaml` creates a YOLO-compatible `data.yaml`.

In [ ]:
import shutil, yaml as _yaml

def remap_labels(src_lbl_dir, dst_lbl_dir, remap):
    """Rewrite label files with class indices remapped according to `remap` dict."""
    dst = Path(dst_lbl_dir); dst.mkdir(parents=True, exist_ok=True)
    n_boxes = 0
    for lp in Path(src_lbl_dir).glob("*.txt"):
        rows = []
        for ln in lp.read_text().splitlines():
            v = ln.split()
            if not v: continue
            rows.append(f"{remap.get(int(v[0]), int(v[0]))} {' '.join(v[1:])}")
            n_boxes += 1
        (dst / lp.name).write_text("\n".join(rows))
    return n_boxes

def link_images(src_img_dir, dst_img_dir):
    """Hardlink images (fast, no extra disk) with copy fallback."""
    dst = Path(dst_img_dir); dst.mkdir(parents=True, exist_ok=True)
    count = 0
    for ip in Path(src_img_dir).glob("*.[jJpP][pPnN][gG]*"):
        d = dst / ip.name
        if not d.exists():
            try:    os.link(ip, d)
            except OSError: shutil.copy2(ip, d)
        count += 1
    return count

def write_yolo_yaml(yaml_path, dataset_root, names_list):
    """dataset_root must have test/images and test/labels inside it."""
    d = {
        "path" : str(Path(dataset_root)),
        "train": "test/images",
        "val"  : "test/images",
        "test" : "test/images",
        "names": {i:n for i,n in enumerate(names_list)},
    }
    Path(yaml_path).write_text(_yaml.dump(d, sort_keys=False))

print("Helpers defined.")

## 7. Build remapped temp datasets

For each cross-evaluation pair the GT labels are rewritten into the evaluating model's index space, then a YOLO yaml is created pointing at that temp dir.

In [ ]:
# ── A) Run1 model evaluating Run2 test data ───────────────────────────────
print("A) Run1 model -> Run2 test  (remapping Run2 labels to Run1 index space)")
base = CROSS_TMP / "run1_on_run2"
ni   = link_images(KAG_TEST_IMGS,  base/"test"/"images")
nb   = remap_labels(KAG_TEST_LBLS, base/"test"/"labels", RUN2_TO_RUN1)
write_yolo_yaml(CROSS_TMP/"run1_on_run2.yaml", base, RUN1_NAMES)
print(f"   {ni} images | {nb} boxes remapped")

# ── B) Run2 model evaluating Run1 test data ───────────────────────────────
print("B) Run2 model -> Run1 test  (remapping Run1 labels to Run2 index space)")
base = CROSS_TMP / "run2_on_run1"
ni   = link_images(RUN1_TEST_IMGS,  base/"test"/"images")
nb   = remap_labels(RUN1_TEST_LBLS, base/"test"/"labels", RUN1_TO_RUN2)
write_yolo_yaml(CROSS_TMP/"run2_on_run1.yaml", base, RUN2_NAMES)
print(f"   {ni} images | {nb} boxes remapped")

# ── C) Run1 model evaluating DeepPCB test (DeepPCB labels are in Run2 space)
if deeppcb_ok:
    print("C) Run1 model -> DeepPCB    (remapping DeepPCB[Run2-space] labels to Run1 index space)")
    base = CROSS_TMP / "run1_on_deeppcb"
    ni   = link_images(DEEPPCB_YOLO/"test"/"images", base/"test"/"images")
    nb   = remap_labels(DEEPPCB_YOLO/"test"/"labels", base/"test"/"labels", RUN2_TO_RUN1)
    write_yolo_yaml(CROSS_TMP/"run1_on_deeppcb.yaml", base, RUN1_NAMES)
    print(f"   {ni} images | {nb} boxes remapped")

print("Done.")

## 8. Run cross-domain evaluations

This is inference-only — no training. Should take a few minutes per cell on a 3090.

In [ ]:
import pandas as pd

def evaluate(model, yaml_path, train_label, test_label, names_list):
    print(f"\n{'='*60}")
    print(f"  {train_label}  ->  {test_label}")
    print(f"{'='*60}")
    r = model.val(
        data    = str(yaml_path),
        split   = "test",
        imgsz   = IMG_SIZE,
        batch   = BATCH,
        device  = DEVICE,
        project = str(RESULTS_DIR),
        name    = f"cross_{train_label}_on_{test_label}",
        exist_ok= True,
        plots   = False,
        verbose = False,
    )
    P, R   = float(r.box.mp), float(r.box.mr)
    m50, m = float(r.box.map50), float(r.box.map)
    idx    = list(r.box.ap_class_index)
    ap50   = {int(c): float(a) for c, a in zip(idx, r.box.ap50)}
    ap     = {int(c): float(a) for c, a in zip(idx, r.box.ap)}
    pc_df  = pd.DataFrame({
        "Class"   : names_list,
        "AP@50"   : [round(ap50.get(i, float("nan")), 4) for i in range(6)],
        "AP@50-95": [round(ap.get(i,   float("nan")), 4) for i in range(6)],
    })
    print(f"  mAP@50={m50:.4f}  mAP@50-95={m:.4f}  P={P:.4f}  R={R:.4f}")
    print(pc_df.to_string(index=False))
    pc_df.to_csv(RESULTS_DIR / f"cross_{train_label}_on_{test_label}_perclass.csv", index=False)
    return {"train":train_label,"test":test_label,
            "P":round(P,4),"R":round(R,4),"mAP50":round(m50,4),"mAP50-95":round(m,4)}

new_results = []
new_results.append(evaluate(model_run1, CROSS_TMP/"run1_on_run2.yaml",     "Run1","Run2",    RUN1_NAMES))
new_results.append(evaluate(model_run2, CROSS_TMP/"run2_on_run1.yaml",     "Run2","Run1",    RUN2_NAMES))
if deeppcb_ok:
    new_results.append(evaluate(model_run1, CROSS_TMP/"run1_on_deeppcb.yaml","Run1","DeepPCB",RUN1_NAMES))

## 9. Full generalization matrix

In [ ]:
def read_indomain(csv_path, split="Test"):
    df  = pd.read_csv(csv_path)
    row = df[df["Split"].str.strip().str.lower() == split.lower()].iloc[0]
    return float(row["mAP@50"]), float(row["mAP@50-95"])

r1r1_m50, r1r1_m = read_indomain(RESULTS_DIR/"exp_001_yolov11n_baseline_200ep_run1_summary.csv")
r2r2_m50, r2r2_m = read_indomain(RESULTS_DIR/"exp_002_yolov11n_kaggle_pcb_200ep_summary.csv")

# Run2->DeepPCB (from run3 notebook)
cdcsv = RESULTS_DIR / "crossdomain_run2_vs_deeppcb.csv"
if cdcsv.exists():
    cd = pd.read_csv(cdcsv)
    row = cd[cd["Setting"].str.contains("cross", case=False)].iloc[0]
    r2dp_m50, r2dp_m = float(row["mAP@50"]), float(row["mAP@50-95"])
else:
    r2dp_m50, r2dp_m = 0.0, 0.0

# Index new results
nr = {(d["train"], d["test"]): d for d in new_results}
def g(tr, te, key): return nr[(tr,te)][key] if (tr,te) in nr else None

# ── Matrix display ──────────────────────────────────────────────────────────
test_sets    = ["Run1 test", "Run2 test", "DeepPCB test"]
train_models = ["Run1", "Run2"]

m50_vals = [
    [r1r1_m50,           g("Run1","Run2","mAP50"),  g("Run1","DeepPCB","mAP50")],
    [g("Run2","Run1","mAP50"), r2r2_m50,            r2dp_m50],
]
m_vals = [
    [r1r1_m,             g("Run1","Run2","mAP50-95"), g("Run1","DeepPCB","mAP50-95")],
    [g("Run2","Run1","mAP50-95"), r2r2_m,             r2dp_m],
]

rows = []
for i, tr in enumerate(train_models):
    for j, te in enumerate(test_sets):
        rows.append({"Train": tr, "Test": te,
                     "mAP@50": m50_vals[i][j], "mAP@50-95": m_vals[i][j]})
mat_df = pd.DataFrame(rows)
mat_df.to_csv(RESULTS_DIR/"generalization_matrix.csv", index=False)
print("\n====  GENERALIZATION MATRIX (mAP@50)  ====")
print(mat_df.pivot(index="Train", columns="Test", values="mAP@50").to_string())
print("\n====  GENERALIZATION MATRIX (mAP@50-95)  ====")
print(mat_df.pivot(index="Train", columns="Test", values="mAP@50-95").to_string())
print("\nSaved: results/generalization_matrix.csv")

## 10. Heatmap visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, vals, title in zip(axes, [m50_vals, m_vals], ["mAP@50", "mAP@50-95"]):
    arr = np.array([[v if v is not None else np.nan for v in row] for row in vals], dtype=float)
    im  = ax.imshow(arr, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(test_sets)));    ax.set_xticklabels(test_sets, fontsize=9)
    ax.set_yticks(range(len(train_models))); ax.set_yticklabels(train_models, fontsize=10, fontweight="bold")
    ax.set_xlabel("Test set"); ax.set_ylabel("Train model")
    ax.set_title(title, fontsize=12, fontweight="bold")
    for i in range(len(train_models)):
        for j in range(len(test_sets)):
            v = arr[i, j]
            txt = f"{v:.3f}" if not np.isnan(v) else "N/A"
            w   = "bold" if i == j else "normal"           # diagonal = in-domain
            ax.text(j, i, txt, ha="center", va="center", fontsize=13, fontweight=w,
                    color="white" if v < 0.35 else "black")
        # bold box around diagonal cell
        ax.add_patch(mpatches.FancyBboxPatch(
            (i - 0.48, i - 0.48), 0.96, 0.96,
            boxstyle="square,pad=0", linewidth=2.5,
            edgecolor="navy", facecolor="none", zorder=5
        ))
    plt.colorbar(im, ax=ax)

plt.suptitle("Cross-Dataset Generalization Matrix
(bold box = in-domain)", fontsize=13, fontweight="bold")
plt.tight_layout()
hm = RESULTS_DIR / "generalization_matrix_heatmap.png"
plt.savefig(hm, dpi=150, bbox_inches="tight"); plt.show()
print("Saved:", hm)

## 11. Interpretation

**Reading the matrix:**
- **Diagonal** (bold border) = in-domain performance. Both models should be high.
- **Off-diagonal Run1↔Run2** = cross-dataset transfer within the same defect modality (color images, same 6 classes). The gap here is the core benchmark finding for same-class, different-source generalization.
- **DeepPCB column** = transfer to a binary-image domain. A large drop here shows the modality sensitivity of current detectors.

**Next steps to complete the paper:**
1. Train a YOLOv11n on DeepPCB and fill in the DeepPCB row (making a proper 3×3 matrix where every diagonal is high).
2. Apply a domain-generalization intervention (strong augmentation / grayscale pre-processing / style transfer) and show per-column recovery.
3. This matrix + the analysis is the core contribution of the benchmark paper.